## HW 2 - Exercise (a)

For a 3D Cartesian grid with `N` points per dimension, the basis size is: $M = N^3$.

The dense Hamiltonian has $M^2 = N^6$ floating-point entries.

If each float uses 8 bytes, the RAM required is approximately: $\text{RAM}(N) \approx 8N^6$ bytes.

## HW 2 - Exercise (b)

For a 3D Cartesian grid, $\hat{H}$ has dimension $M\times M$ with $M=N^3$.
Assuming 7 nonzeros per row:
$$\text{nnz} \approx 7M = 7N^3$$

CSR stores three arrays:
- `data`: `nnz` floats $\Rightarrow 7N^3\,b_f$ bytes
- `indices`: `nnz` integers $\Rightarrow 7N^3\,b_i$ bytes
- `indptr`: `M+1=N^3+1` integers $\Rightarrow (N^3+1)\,b_i$ bytes

So the total RAM is:
$$\text{RAM}_{\text{CSR}} \approx 7N^3\,b_f + 7N^3\,b_i + (N^3+1)\,b_i = 7N^3\,b_f + (8N^3+1)\,b_i\ \text{bytes}.$$

If `data=float64` and integer arrays are `int32`:
$$\text{RAM}_{\text{CSR}} \approx 7N^3\cdot 8 + (8N^3+1)\cdot 4 = 88N^3 + 4\ \text{bytes}.$$

## HW 2 - Exercise (c)

The following code plots memory (GB) for dense and sparse storage over $N\in[10,200]$, and finds the smallest $N$ where each exceeds 16 GB.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# N range
N = np.arange(10, 201)

# Byte-size assumptions
b_float = 8  # float64
b_int = 4    # int32
laptop_bytes = 16 * (1024**3)  # 16 GiB

# Memory models in bytes
dense_bytes = 8 * N**6
sparse_bytes = 7 * N**3 * b_float + (8 * N**3 + 1) * b_int  # = 88*N^3 + 4

# Convert to GB (decimal)
to_gb = 1e9
dense_gb = dense_bytes / to_gb
sparse_gb = sparse_bytes / to_gb

# Critical N: smallest integer N where memory exceeds 16 GiB
N_dense_crit = int(np.ceil((laptop_bytes / 8) ** (1/6)))
N_sparse_crit = int(np.ceil(((laptop_bytes - 4) / 88) ** (1/3)))

print(f'Dense exceeds 16 GiB at N >= {N_dense_crit}')
print(f'Sparse (CSR) exceeds 16 GiB at N >= {N_sparse_crit}')

plt.figure(figsize=(8, 5))
plt.plot(N, dense_gb, label='Dense H (8N^6 bytes)', linewidth=2)
plt.plot(N, sparse_gb, label='Sparse CSR H (88N^3+4 bytes)', linewidth=2)
plt.axhline(laptop_bytes / to_gb, color='k', linestyle='--', label='16 GiB laptop RAM')
plt.axvline(N_dense_crit, color='tab:blue', linestyle=':', alpha=0.8)
if 10 <= N_sparse_crit <= 200:
    plt.axvline(N_sparse_crit, color='tab:orange', linestyle=':', alpha=0.8)

plt.xlabel('N')
plt.ylabel('Memory (GB)')
plt.title('Dense vs Sparse Memory Requirement')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## HW 2 - Exercise (d)

Build the dense 3D kinetic operator using Kronecker products of 1D $\hat{T}$ and identity matrices for several $N\leq 13$. Then compare measured memory from `T.nbytes` to the algebraic scaling from (a), $8N^6$ bytes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hbar = 1.0
m = 1.0

def T1D_dense(N):
    """1D kinetic operator with central-difference stencil, using hbar=1, m=1."""
    main = -2.0 * np.ones(N)
    off = 1.0 * np.ones(N - 1)
    lap1d = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)
    return -(hbar**2 / (2.0 * m)) * lap1d

N_vals = np.arange(2, 14)  # N <= 13
actual_bytes = []
model_bytes = []

for N in N_vals:
    T = T1D_dense(N)
    I = np.eye(N)

    T3 = np.kron(np.kron(T, I), I) + np.kron(np.kron(I, T), I) + np.kron(np.kron(I, I), T)

    actual_bytes.append(T3.nbytes)
    model_bytes.append(8 * N**6)

actual_bytes = np.array(actual_bytes)
model_bytes = np.array(model_bytes)

print('N, actual bytes, model bytes (8N^6):')
for N, a, b in zip(N_vals, actual_bytes, model_bytes):
    print(f'{N:2d}: {a:12d}  {b:12d}')

plt.figure(figsize=(8, 5))
plt.plot(N_vals, actual_bytes / 1e9, 'o-', label='Measured (T3.nbytes)')
plt.plot(N_vals, model_bytes / 1e9, 's--', label='Model from (a): 8N^6 bytes')
plt.xlabel('N')
plt.ylabel('Memory (GB)')
plt.title('Dense 3D $\\hat{T}$ Memory: Measured vs Algebraic')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()